# EDA 테스트 파일 (CatBoost 기준)

- EDA를 수정하면서 ACC / F1 / AUC 수치로 효과를 빠르게 확인하는 파일
- CatBoost + Optuna 튜닝 → 20% Val 비교
- 앙상블 없음 (단일 모델 집중)

## 0. 환경 설정

In [ ]:
# ── 패키지 설치 ───────────────────────────────────────────────────────────────
!pip install -q catboost optuna

# ── 라이브러리 임포트 ─────────────────────────────────────────────────────────
import os, subprocess, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.impute import SimpleImputer

from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── 전역 설정 ─────────────────────────────────────────────────────────────────
SEED      = 42
N_TRIALS  = 50   # Optuna 탐색 횟수
DATA_PATH = '/content/drive/MyDrive/open'

np.random.seed(SEED)

# ── 한글 폰트 설정 ────────────────────────────────────────────────────────────
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fe = fm.FontEntry(fname=font_path, name='NanumGothic')
fm.fontManager.ttflist.insert(0, fe)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print('환경 설정 완료 ✓')

## 1. 데이터 로드

In [ ]:
train = pd.read_csv(f'{DATA_PATH}/train.csv')

print(f'train shape: {train.shape}')
print(f'컬럼 수    : {train.shape[1]}')
print(f'타겟 분포  :\n{train["completed"].value_counts()}')
train.head(3)

## 2. EDA

> **이 섹션을 수정하면서 ACC/F1/AUC 변화를 확인하세요.**
> - 새로운 피처 추가/제거, 이상치 처리, 인코딩 방식 변경 등을 여기서 시도합니다.

In [ ]:
# ── 기본 정보 ─────────────────────────────────────────────────────────────────
print('=== 데이터 타입 & 결측치 ===')
info_df = pd.DataFrame({
    'dtype'   : train.dtypes,
    'null_cnt': train.isnull().sum(),
    'null_pct': (train.isnull().mean() * 100).round(2),
    'nunique' : train.nunique()
})
display(info_df)
print(f'\n결측 컬럼 수: {(train.isnull().sum() > 0).sum()}')

In [ ]:
# ── 타겟 분포 ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

vc = train['completed'].value_counts()
axes[0].bar(vc.index.astype(str), vc.values, color=['#5B9BD5', '#ED7D31'])
axes[0].set_title('Target 분포 (completed)')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=12)

axes[1].pie(vc.values, labels=['미완료(0)', '완료(1)'], autopct='%1.1f%%',
            colors=['#5B9BD5', '#ED7D31'], startangle=90)
axes[1].set_title('Target 비율')

plt.tight_layout()
plt.show()
print(vc)

In [ ]:
# ── 수치형 피처 분포 ──────────────────────────────────────────────────────────
num_cols = train.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c not in ['ID', 'completed']]
print(f'수치형 컬럼 ({len(num_cols)}개): {num_cols}')

if num_cols:
    fig, axes = plt.subplots(len(num_cols), 2, figsize=(12, 4 * len(num_cols)))
    if len(num_cols) == 1:
        axes = [axes]
    for i, col in enumerate(num_cols):
        train[col].hist(ax=axes[i][0], bins=30, color='steelblue', edgecolor='white')
        axes[i][0].set_title(f'{col} 분포')
        train.boxplot(column=col, by='completed', ax=axes[i][1])
        axes[i][1].set_title(f'{col} by completed')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 카테고리형 피처 빈도 ──────────────────────────────────────────────────────
cat_cols = train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'ID']
print(f'카테고리형 컬럼 수: {len(cat_cols)}')

if cat_cols:
    fig, axes = plt.subplots(len(cat_cols), 1, figsize=(14, 4 * len(cat_cols)))
    if len(cat_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_cols):
        vc = train[col].value_counts().head(10)
        ax.barh(vc.index.astype(str), vc.values, color='steelblue')
        ax.set_title(f'{col} (top 10)')
        ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 상관관계 히트맵 ───────────────────────────────────────────────────────────
num_cols_corr = train.select_dtypes(include='number').columns.tolist()
num_cols_corr = [c for c in num_cols_corr if c != 'ID']

if len(num_cols_corr) > 1:
    corr = train[num_cols_corr].corr()
    plt.figure(figsize=(max(8, len(num_cols_corr)), max(6, len(num_cols_corr) - 1)))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
    plt.title('상관관계 히트맵')
    plt.tight_layout()
    plt.show()

## 3. 피처 엔지니어링 (수정 포인트)

> **여기서 새 피처를 추가하거나 기존 피처를 변환하세요.**
> - 예: 로그 변환, 구간화(binning), 파생 피처 생성 등

In [ ]:
# ── 피처 엔지니어링 (필요 시 수정) ───────────────────────────────────────────
# 예시: 새 파생 피처 추가
# train['new_feature'] = train['col_a'] / (train['col_b'] + 1)

# 제거할 컬럼 지정 (필요 시 수정)
DROP_COLS = ['ID']   # 기본값: ID만 제거

print(f'현재 컬럼 수: {train.shape[1]}')
print(f'제거할 컬럼: {DROP_COLS}')
print(f'피처 엔지니어링 완료 ✓')

## 4. 전처리

In [ ]:
def preprocess(df, drop_cols=None):
    """단일 데이터프레임 전처리 (train 전용)"""
    if drop_cols is None:
        drop_cols = ['ID']

    y = df['completed'].copy()
    X = df.drop(columns=drop_cols + ['completed'], errors='ignore').copy()

    # bool → int
    for col in X.select_dtypes(include='bool').columns:
        X[col] = X[col].astype(int)

    # 'True'/'False' 문자열 → int
    for col in X.select_dtypes(include='object').columns:
        unique_vals = set(X[col].dropna().unique())
        if unique_vals <= {'True', 'False'}:
            X[col] = X[col].map({'True': 1, 'False': 0})

    # 카테고리형 → Label Encoding
    cat_cols = X.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

    # 결측치 처리 (수치형 median)
    num_cols = X.select_dtypes(include='number').columns.tolist()
    if num_cols:
        imputer = SimpleImputer(strategy='median', keep_empty_features=True)
        X[num_cols] = imputer.fit_transform(X[num_cols])

    return X, y

X, y = preprocess(train, drop_cols=DROP_COLS)
print(f'X shape: {X.shape}')
print(f'y 분포 :\n{y.value_counts()}')

## 5. Train / Validation 분리 (8:2)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f'X_train: {X_train.shape}  |  y_train 분포: {dict(y_train.value_counts())}')
print(f'X_val  : {X_val.shape}  |  y_val   분포: {dict(y_val.value_counts())}')

## 6. CatBoost 베이스라인 (빠른 확인용)

In [ ]:
# ── 베이스라인 (튜닝 전) ──────────────────────────────────────────────────────
base_cat = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    random_seed=SEED,
    verbose=0
)
base_cat.fit(X_train, y_train)

pred_base = base_cat.predict(X_val)
prob_base = base_cat.predict_proba(X_val)[:, 1]

acc_base = accuracy_score(y_val, pred_base)
f1_base  = f1_score(y_val, pred_base)
auc_base = roc_auc_score(y_val, prob_base)

print('=' * 50)
print('[ CatBoost 베이스라인 Val 성능 ]')
print('=' * 50)
print(f'  ACC : {acc_base:.4f}')
print(f'  F1  : {f1_base:.4f}')
print(f'  AUC : {auc_base:.4f}')

## 7. CatBoost + Optuna 하이퍼파라미터 튜닝

In [ ]:
def cat_objective(trial):
    params = {
        'iterations'         : trial.suggest_int('iterations', 200, 1000),
        'learning_rate'      : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth'              : trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg'        : trial.suggest_float('l2_leaf_reg', 1e-4, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'border_count'       : trial.suggest_int('border_count', 32, 255),
        'random_strength'    : trial.suggest_float('random_strength', 1e-4, 10.0, log=True),
        'random_seed'        : SEED,
        'verbose'            : 0,
    }
    m = CatBoostClassifier(**params)
    m.fit(X_train, y_train)
    return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

cat_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
cat_study.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n[CatBoost 튜닝 완료]  Best AUC: {cat_study.best_value:.4f}')
print(f'Best params: {cat_study.best_params}')

## 8. 튜닝된 모델 최종 평가

In [ ]:
# ── 최적 파라미터로 재학습 ────────────────────────────────────────────────────
cat_tuned = CatBoostClassifier(**cat_study.best_params, verbose=0)
cat_tuned.fit(X_train, y_train)

pred_tuned = cat_tuned.predict(X_val)
prob_tuned = cat_tuned.predict_proba(X_val)[:, 1]

acc_tuned = accuracy_score(y_val, pred_tuned)
f1_tuned  = f1_score(y_val, pred_tuned)
auc_tuned = roc_auc_score(y_val, prob_tuned)

# ── 결과 비교표 ───────────────────────────────────────────────────────────────
result_df = pd.DataFrame([
    {'모델': 'CatBoost [Baseline]', 'ACC': acc_base,  'F1': f1_base,  'AUC': auc_base},
    {'모델': 'CatBoost [Tuned]',    'ACC': acc_tuned, 'F1': f1_tuned, 'AUC': auc_tuned},
])

print('=' * 55)
print('[ CatBoost 베이스라인 vs 튜닝 비교 (20% Val) ]')
print('=' * 55)
display(result_df.style
        .highlight_max(subset=['ACC','F1','AUC'], color='#c6efce')
        .format({'ACC': '{:.4f}', 'F1': '{:.4f}', 'AUC': '{:.4f}'}))

print(f'\n성능 향상:')
print(f'  ACC : {acc_base:.4f} → {acc_tuned:.4f}  ({acc_tuned - acc_base:+.4f})')
print(f'  F1  : {f1_base:.4f} → {f1_tuned:.4f}  ({f1_tuned - f1_base:+.4f})')
print(f'  AUC : {auc_base:.4f} → {auc_tuned:.4f}  ({auc_tuned - auc_base:+.4f})')

## 9. 상세 분석 (Classification Report + Confusion Matrix)

In [ ]:
print('[ Classification Report — CatBoost Tuned ]')
print(classification_report(y_val, pred_tuned, target_names=['미완료(0)', '완료(1)']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix
cm = confusion_matrix(y_val, pred_tuned)
ConfusionMatrixDisplay(cm, display_labels=['미완료', '완료']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix\nCatBoost [Tuned]  ACC={acc_tuned:.4f}')

# Feature Importance
fi = pd.Series(
    cat_tuned.get_feature_importance(),
    index=X_train.columns
).sort_values(ascending=True)
fi.tail(20).plot(kind='barh', ax=axes[1], color='#ED7D31')
axes[1].set_title('Feature Importance Top 20 (CatBoost Tuned)')

plt.tight_layout()
plt.show()

## 10. EDA 수정 전후 비교 기록

> 아래 셀에 EDA 수정 결과를 누적 기록하세요.

In [ ]:
# ── EDA 버전별 결과 누적 기록 ─────────────────────────────────────────────────
# 새로운 결과가 나올 때마다 아래 리스트에 추가하세요
history = [
    # {'version': 'v1 기본 전처리', 'ACC': 0.0000, 'F1': 0.0000, 'AUC': 0.0000, '메모': ''},
]

# 이번 실행 결과 자동 추가
history.append({
    'version': f'현재 실행 (Tuned)',
    'ACC'    : round(acc_tuned, 4),
    'F1'     : round(f1_tuned,  4),
    'AUC'    : round(auc_tuned, 4),
    '메모'   : '기본 전처리'
})

history_df = pd.DataFrame(history)
display(history_df.style
        .highlight_max(subset=['ACC','F1','AUC'], color='#c6efce')
        .format({'ACC': '{:.4f}', 'F1': '{:.4f}', 'AUC': '{:.4f}'}))

# 시각화
if len(history_df) > 1:
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(history_df))
    w = 0.25
    ax.bar(x - w, history_df['ACC'], w, label='ACC',  color='#5B9BD5')
    ax.bar(x,     history_df['F1'],  w, label='F1',   color='#ED7D31')
    ax.bar(x + w, history_df['AUC'], w, label='AUC',  color='#70AD47')
    ax.set_xticks(x)
    ax.set_xticklabels(history_df['version'], rotation=20, ha='right')
    ax.set_ylim(0.5, 1.05)
    ax.legend()
    ax.set_title('EDA 버전별 성능 비교')
    plt.tight_layout()
    plt.show()